# MSDS Course Time Estimator (v3)

**Pipeline** (`modules/pipeline.py`):
1. Bayesian posterior on each course's true mean (`data_points.csv` where `include='true'`).
2. Anchor-course speed `S`.
3. Skill decomposition → per-course `S_target`.
4. Focus-ratio + ADHD variance scaling.
5. Lognormal marginals for project-heavy courses.

See `LITERATURE.md` for citations.


In [ ]:
%load_ext autoreload
%autoreload 2

import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd()))
import pandas as pd
from modules import pipeline, form_intake


## 1. Pull fresh crowdsourced data (optional)

In [ ]:
if form_intake.PUBLISHED_CSV_URL:
    stats = form_intake.sync()
    print(f"Pulled {stats['total_seen']} submissions; {stats['added_auto']} new auto-approved, {stats['added_review']} new flagged.")
else:
    print("PUBLISHED_CSV_URL not set; skipping.")

review_df = pd.read_csv('data_points_review.csv', dtype=str).fillna('')
pending = review_df[review_df['include'] == 'pending']
print(f"Pending review rows: {len(pending)}")
pending[['submission_id','course_id','days','hours_per_week','concurrent_courses','has_adhd','notes']]


### Approve flagged rows

Fill in decisions, run to commit. Only `'true'` rows flow into the posterior.

In [ ]:
decisions = {
    # 'form_20260101T120000': 'true',
    # 'form_20260101T130000': 'false',
}
if decisions:
    n = form_intake.promote_reviewed(decisions)
    print(f"Promoted {n} rows. Re-run the next cell to refresh the posterior.")


## 2. Your profile

In [ ]:
USE_PROMPTS = True

if USE_PROMPTS:
    inputs = pipeline.prompt_profile()
else:
    # Put completed courses here. Values are calendar days to finish.
    # Keep 0.0 for courses that should not be used as anchors.
    anchor_days_by_course = {
    "DTSA5001": 46.0,

    "DTSA5002": 0.0,

    "DTSA5003": 0.0,

    "DTSA5501": 0.0,

    "DTSA5502": 0.0,

    "DTSA5503": 0.0,

    "DTSA5011": 0.0,

    "DTSA5012": 0.0,

    "DTSA5013": 0.0,

    "DTSA5504": 0.0,

    "DTSA5505": 0.0,

    "DTSA5506": 0.0,

    "DTSA5509": 0.0,

    "DTSA5510": 0.0,

    "DTSA5511": 0.0,

    "DTSA5733": 0.0,

    "DTSA5734": 0.0,

    "DTSA5735": 0.0,

    "DTSA5301": 0.0,

    "DTSA5302": 0.0,

    "DTSA5303": 0.0,

    "DTSA5304": 0.0,

    "DTSA5020": 0.0,

    "DTSA5021": 0.0,

    "DTSA5022": 0.0,

    "DTSA5701": 0.0,

    "DTSA5702": 0.0,

    "DTSA5703": 0.0,

    "DTSA5704": 0.0,

    "DTSA5705": 0.0,

    "DTSA5706": 0.0,

    "DTSA5798": 0.0,

    "DTSA5799": 0.0,

    "DTSA5800": 0.0,

    "DTSA5842": 0.0,

    "DTSA5843": 0.0,
    }
    anchor_courses, anchor_days_list = pipeline.anchors_from_days_dict(
        anchor_days_by_course,
        post_params["course_id"].tolist(),
    )
    inputs = dict(
        anchor_courses=anchor_courses,
        anchor_days_list=anchor_days_list,
        user_focus_ratio=0.70,
        has_adhd=False,
        medicated=False,
        skill_priors=(5, 5, 5),
        anchor_hours_per_week=10.0,
        target_hours_per_week=10.0,
        concurrent_courses=1,
        courses_completed=0,
        anchor_concurrent_courses=1,
        anchor_courses_completed=0,
    )

profile = pipeline.build_profile(post_params=post_params, **inputs)
pipeline.print_profile(profile)


## 3. Run the pipeline

In [ ]:
post_params = pipeline.load_posterior_params()
# split target/deadline out so build_profile only gets its kwargs
profile_kwargs = {k: v for k, v in inputs.items() if k not in ('target_course','deadline_days','risk_tolerance')}
TARGET = inputs['target_course']
DEADLINE_DAYS = inputs.get('deadline_days', 56)
profile = pipeline.build_profile(post_params, **profile_kwargs)
pipeline.print_profile(profile)

predictions = pipeline.build_predictions(post_params, profile)
predictions[['course_id','name','prior_days','predicted_days','sd','dist_type','80%_interval']]


## 4. Visualize one course

In [ ]:
_ = pipeline.plot_course(predictions, TARGET, profile)


## 5. Deadline probability

In [ ]:
RISK = inputs.get('risk_tolerance', 'med')
p, safe = pipeline.prob_finish(predictions, TARGET, DEADLINE_DAYS, RISK)
print(f'P(finish {TARGET} in <= {DEADLINE_DAYS:.0f} days) = {p*100:.1f}%')
print(f'Safe buffer ({RISK} risk tolerance): plan for {safe:.0f} days')


## 6. Degree-plan total

Select the 30-33 courses you actually plan to take, then aggregate with CLT using correlated variance. Do not sum every modeled course.


In [ ]:
DEGREE_COURSE_IDS = [
    # Required/pathway courses
    "DTSA5001", "DTSA5002", "DTSA5003",
    "DTSA5011", "DTSA5012", "DTSA5013",
    "DTSA5501", "DTSA5502", "DTSA5503",
    "DTSA5504", "DTSA5505", "DTSA5506",
    "DTSA5509", "DTSA5510", "DTSA5511",
    "DTSA5733", "DTSA5734", "DTSA5735",
    "DTSA5301", "DTSA5302", "DTSA5303", "DTSA5304",
    # Add/remove electives here until this matches your actual 30-33 course plan
    "DTSA5020", "DTSA5021", "DTSA5022",
    "DTSA5701", "DTSA5702", "DTSA5703",
    "DTSA5704", "DTSA5705", "DTSA5706",
]

try:
    degree_predictions = pipeline.filter_degree_plan(predictions, DEGREE_COURSE_IDS)
    total = pipeline.predict_degree_total(degree_predictions, post_params, risk_tolerance=RISK)
    print(f"Selected {len(DEGREE_COURSE_IDS)} courses for the degree plan.")
    print(f"Degree-plan estimate ({total['n_courses']} courses):")
    print(f"  Mean:       {total['total_mu']:.0f} days  (~{total['total_mu']/30.4:.1f} months)")
    print(f"  80% range:  {total['p10']:.0f} - {total['p90']:.0f} days")
    print(f"  Safe buffer ({RISK} risk): plan for {total['safe_days']:.0f} days  (~{total['safe_days']/30.4:.1f} months)")
    print(f"  sd = {total['total_sd']:.0f} days (topic corr {total['topic_base_corr']:.2f}-{total['topic_max_corr']:.2f})")
except ValueError as exc:
    print(exc)


## Notes

- `modules/pipeline.py` owns the orchestration; the other modules own their own concerns.
- `data_points.csv` is the single source of truth; `include` column gates inclusion.
- To skip interactive prompts, uncomment the hardcoded `inputs` block above.
